In [ ]:
# =======================
# STEP 1: INSTALLATIONS
# =======================
!apt update
!apt install poppler-utils
!apt install -y tesseract-ocr tesseract-ocr-eng tesseract-ocr-hin
!pip install pytesseract pdf2image pillow opencv-python numpy reportlab

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,775 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,269 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9

In [ ]:
# =======================
# STEP 2: IMPORTS
# =======================
import pytesseract
from pdf2image import convert_from_path
from PIL import Image, ImageEnhance
import cv2
import numpy as np
import os
import textwrap
from google.colab import files

from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from datetime import datetime

In [ ]:
# =======================
# OCR on PDF or Image and Save as PDF
# =======================

def enhance_image(image):
    img = image.convert('L')
    img = img.resize((img.width * 2, img.height * 2))
    enhancer = ImageEnhance.Contrast(img)
    img = enhancer.enhance(2)
    img = img.point(lambda x: 0 if x < 140 else 255)
    return img

def extract_mixed_text_from_pdf(pdf_path):
    # Convert PDF to images
    pages = convert_from_path(pdf_path, dpi=300)
    full_text = ""

    for i, page in enumerate(pages):
        print(f"🔍 OCR on page {i+1}")
        img = enhance_image(page)
        text = pytesseract.image_to_string(img, lang='eng+hin', config='--psm 11')
        full_text += text + "\n\n"

    return full_text.strip()

def save_text_as_pdf(text, output_path, title="Extracted Text"):
    font_path = "Mukta-Regular.ttf"
    font_name = "Mukta"

    if not os.path.exists(font_path):
        print("📥 Downloading Mukta font...")
        os.system("wget -O Mukta-Regular.ttf https://github.com/google/fonts/raw/main/ofl/mukta/Mukta-Regular.ttf")

    pdfmetrics.registerFont(TTFont(font_name, font_path))

    # Create the PDF
    c = canvas.Canvas(output_path, pagesize=A4)
    width, height = A4
    c.setTitle(title)
    c.setFont(font_name, 12)

    x = 50
    y = height - 50
    line_height = 18
    max_chars = 90

    for line in text.split('\n'):
        wrapped = textwrap.wrap(line.strip(), width=max_chars)
        for wline in wrapped:
            if y < 50:
                c.showPage()
                c.setFont(font_name, 12)
                y = height - 50
            c.drawString(x, y, wline)
            y -= line_height

    c.save()
    print(f"✅ Unicode bilingual PDF saved as: {output_path}")

# File upload
uploaded_file = files.upload()
if not uploaded_file:
    raise ValueError("No file uploaded!")

file_name = list(uploaded_file.keys())[0]

extracted_text = ""

try:
    if file_name.lower().endswith('.pdf'):
        print(f"Processing PDF file: {file_name}")
        pages = convert_from_path(file_name, dpi=300)
        for i, page in enumerate(pages):
            print(f"🔍 OCR on page {i+1}")
            img = enhance_image(page)
            text = pytesseract.image_to_string(img, lang='eng+hin', config='--psm 3')
            extracted_text += text + "\n\n"

    elif file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff', '.bmp')):
        print(f"Processing image file: {file_name}")
        img = Image.open(file_name)
        enhanced_img = enhance_image(img)
        extracted_text = pytesseract.image_to_string(enhanced_img, lang='eng+hin', config='--psm 3')

    else:
        raise ValueError("Unsupported file type. Please upload a PDF or an image file.")

    print("\n--- Extracted Text ---")
    print(extracted_text)
    print("----------------------")

    #file download
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_pdf_name = f"{os.path.splitext(file_name)[0]}_extracted_{timestamp}.pdf"
    save_text_as_pdf(extracted_text, output_pdf_name, title=f"Extracted Text from {file_name}")
    files.download(output_pdf_name)

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    # Clean up the uploaded file
    if os.path.exists(file_name):
      os.remove(file_name)

Saving test_img.pdf to test_img.pdf
Processing PDF file: test_img.pdf
🔍 OCR on page 1

--- Extracted Text ---
Government of Indla
Ministry of Micro, Small & Medium Enterprises
O/d the Development Commissioner (MEME)
Nirman ove A-Wing. 7 Floor... 2 ee wee - -- oe
em ee eres ee“ Matilana Azad Road,
- Naw Delhl-110108
Tel, 011-23061001
Fax No.011-23060836

Polley Clreular No. 1(2)(4)/2016-MA 01, 107 March 2016
To
All Central Ministries/Departments/CPSUs/All Concerned

Subject: Relaxation of Norms for Startups and Micro & Small Enterprises in
Public Procurement on Prior Experience — Prior Turnover critérta.

(1) The Government of India has notified Public Procurement Policy for Micro and
Small Enterprises (MSEs) Order 2012 with effect from 1 April, 2012 and 20%
procurement from Micro & Small Enterprises of the total procurement by Central
Ministries/Departments/CPSUs has become mandatory with effect from 1" April,
2015. . OO |

(2) The Government of India has announced ‘Startup India’ init

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>